<a href="https://colab.research.google.com/github/binsue0/.github/blob/main/DL_day3/3_1_approximation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReLU로 곡선 근사하기 — 뉴런·층 쌓기 실습

**핵심 질문:** 신경망은 어떻게 구불구불한 곡선을 표현할까?

이 실습에서는 ** 목표 곡선 하나**를 신경망으로 근사시켜 보면서  확인합니다.

| 실험 | 바꾸는 것 | 확인할 직관 |
|---|---|---|
| **A. 뉴런 1개** | 은닉 유닛 1개 | 직선(꺾임 하나)밖에 못 그림 |
| **B. 뉴런 늘리기** | 은닉 유닛 3 → 10 → 50 | 꺾임(선형 조각)이 많아질수록 곡선에 가까워짐 |
| **C. 깊게 쌓기** | 층을 2개로 | 적은 뉴런으로도 표현력이 커짐 |

> 앞에서 배운 것: **ReLU 하나 = 꺾임 하나.** 꺾인 직선 조각들을 이어붙여 곡선을 근사한다.


## 1. 준비 — 라이브러리

> **Colab에서 한글 그래프가 깨진다면** 아래 셀을 먼저 실행하세요. (로컬에 이미 한글 폰트가 있으면 생략 가능)

In [ ]:
# Colab용: 한글 폰트 (필요할 때만)
!pip install koreanize-matplotlib -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

try:
    import koreanize_matplotlib   # 한글 폰트 자동 적용
except ImportError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
np.random.seed(0)

## 2. 목표 곡선 만들기

근사할 대상은 **입력 x 하나 → 출력 y 하나**인 1차원 곡선입니다.
일부러 구불구불하게(사인 + 약간의 굴곡) 만들어서, 직선으로는 절대 못 맞추게 합니다.

In [ ]:
# 입력 x: -3 ~ 3 사이 200개 점
x = np.linspace(-3, 3, 200).reshape(-1, 1)

# 목표 곡선 (구불구불): 이걸 신경망이 따라 그려야 함
y = np.sin(1.5 * x) + 0.3 * x

# 텐서로 변환
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

plt.figure(figsize=(6, 4))
plt.plot(x, y, 'k--', label='target curve (목표)')
plt.legend(); plt.title('근사할 목표 곡선'); plt.xlabel('x'); plt.ylabel('y'); plt.show()

## 3. 모델 정의

구조는 아주 단순한 MLP. **은닉 유닛 개수(`hidden`)**와 **층 수**만 바꿔가며 비교합니다.

- 활성화 함수는 **ReLU** — 꺾임을 만드는 장본인.
- 은닉 유닛 하나가 꺾임 하나를 담당한다는 걸 기억하세요.

In [ ]:
class ShallowNet(nn.Module):
    """은닉층 1개짜리 얕은 망. hidden 개수만 바꿔가며 실험."""
    def __init__(self, hidden):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden)   # 입력 1개 -> 은닉 유닛 hidden개
        self.fc2 = nn.Linear(hidden, 1)   # 은닉 -> 출력 1개

    def forward(self, x):
        x = torch.relu(self.fc1(x))       # ReLU로 꺾임 생성
        x = self.fc2(x)                   # 꺾인 조각들을 가중합
        return x


class DeepNet(nn.Module):
    """은닉층 2개짜리 깊은 망. 같은 유닛 수라도 표현력이 큰지 비교."""
    def __init__(self, hidden):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### 학습 함수

목표 곡선에 맞추도록 MSE loss로 학습시키고, 학습된 예측 곡선을 돌려줍니다.
(회귀 문제라 loss는 **평균제곱오차(MSE)** — 앞에서 배운 "정규분포 가정 → MSE" 기억나죠?)

In [ ]:
def train_and_predict(model, epochs=2000, lr=0.01):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(x_t)
        loss = criterion(pred, y_t)
        loss.backward()
        optimizer.step()

    # 학습 끝난 뒤 예측 곡선 반환
    with torch.no_grad():
        y_pred = model(x_t).numpy()
    return y_pred, loss.item()

## 실험 A. 뉴런 1개 — 직선밖에 못 긋는다

은닉 유닛을 **1개**만 두면 꺾임이 하나뿐. 목표 곡선을 절대 따라올 수 없습니다.

In [ ]:
model_1 = ShallowNet(hidden=1)
y_pred_1, loss_1 = train_and_predict(model_1)

plt.figure(figsize=(6, 4))
plt.plot(x, y, 'k--', label='target')
plt.plot(x, y_pred_1, 'r-', label='hidden=1 예측')
plt.legend(); plt.title(f'뉴런 1개  (loss={loss_1:.3f})'); plt.show()

print(f'뉴런 1개 loss: {loss_1:.4f}  <- 크다! 곡선을 못 맞춤')

**관찰 포인트:** 예측선이 거의 직선(또는 꺾임 하나)이라 목표 곡선과 크게 어긋납니다.
뉴런 하나의 capacity가 얼마나 작은지 눈으로 확인되죠.

## 실험 B. 뉴런을 늘려보자 — 꺾임이 많아질수록 곡선에 가까워진다

은닉 유닛을 **3 → 10 → 50**으로 늘려가며 예측 곡선을 비교합니다.
유닛이 많아질수록 직선 조각(꺾임)이 촘촘해져서 곡선을 잘 흉내 냅니다.

In [ ]:
hidden_sizes = [3, 10, 50]
results = {}

for h in hidden_sizes:
    model = ShallowNet(hidden=h)
    y_pred, loss = train_and_predict(model)
    results[h] = (y_pred, loss)
    print(f'hidden={h:2d}  ->  loss={loss:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, h in zip(axes, hidden_sizes):
    y_pred, loss = results[h]
    ax.plot(x, y, 'k--', label='target')
    ax.plot(x, y_pred, 'b-', label=f'hidden={h}')
    ax.set_title(f'뉴런 {h}개  (loss={loss:.3f})')
    ax.legend()
plt.suptitle('은닉 유닛을 늘릴수록 곡선에 가까워진다', y=1.02)
plt.tight_layout(); plt.show()

**관찰 포인트:**
- `hidden=3`: 큼직한 직선 조각 몇 개로 대충 흉내
- `hidden=10`: 꽤 비슷해짐
- `hidden=50`: 거의 완벽하게 겹침

바로 **"은닉 유닛 하나당 선형 영역 하나가 추가된다"**를 눈으로 본 것.
그리고 유닛만 충분하면 어떤 곡선이든 근사할 수 있다는 **Universal Approximation**의 직관이기도 합니다.

### 꺾임을 직접 확인하기 (보너스)

`hidden=10` 예측 곡선을 확대해보면, 곡선처럼 보이지만 사실 **직선 조각들이 이어진 것**임을 알 수 있습니다.

In [ ]:
y_pred_10, _ = results[10]

plt.figure(figsize=(7, 4))
plt.plot(x, y_pred_10, 'b-', linewidth=2, label='hidden=10 예측')
plt.plot(x, y, 'k--', alpha=0.4, label='target')
plt.title('확대해서 보면 = 직선 조각들의 이음 (piecewise linear)')
plt.legend(); plt.show()

print('부드러워 보여도 실제로는 여러 개의 직선 조각!')

## 실험 C. 깊게 쌓으면? — 적은 뉴런으로 더 큰 표현력

층을 하나 더 쌓은 **깊은 망**과 같은 유닛 수의 **얕은 망**을 비교합니다.
층 수가 늘면 앞 층의 꺾임이 뒷 층에서 복제·조합되어(folding) 표현력이 커집니다.

In [ ]:
h = 8  # 일부러 작게 잡음

# 얕은 망 (은닉층 1개)
shallow = ShallowNet(hidden=h)
y_shallow, loss_shallow = train_and_predict(shallow)

# 깊은 망 (은닉층 2개, 같은 유닛 수)
deep = DeepNet(hidden=h)
y_deep, loss_deep = train_and_predict(deep)

print(f'얕은 망 (hidden={h}, 1층)  loss={loss_shallow:.4f}')
print(f'깊은 망 (hidden={h}, 2층)  loss={loss_deep:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(x, y, 'k--', label='target')
axes[0].plot(x, y_shallow, 'g-', label='shallow')
axes[0].set_title(f'얕은 망 1층  (loss={loss_shallow:.3f})')
axes[0].legend()

axes[1].plot(x, y, 'k--', label='target')
axes[1].plot(x, y_deep, 'm-', label='deep')
axes[1].set_title(f'깊은 망 2층  (loss={loss_deep:.3f})')
axes[1].legend()

plt.suptitle(f'같은 뉴런 수(hidden={h})라도 깊게 쌓으면 표현력이 커진다', y=1.02)
plt.tight_layout(); plt.show()

**관찰 포인트:** 유닛 수는 같은데, 층을 하나 더 쌓은 깊은 망이 곡선을 더 잘 맞추는 경향이 있습니다.
(랜덤 초기화라 실행마다 조금씩 다를 수 있으니 여러 번 돌려보세요.)

이게 **"깊이의 효율성"** — 넓게 늘리면 꺾임이 더해지고(선형), 깊게 쌓으면 곱해진다(지수).

## 정리

- **뉴런 1개** = 꺾임 1개 = 직선. capacity가 작아서 곡선을 못 맞춤.
- **은닉 유닛을 늘리면** 꺾인 직선 조각이 많아져 → 어떤 곡선이든 근사 가능 (Universal Approximation).
- **깊게 쌓으면** 같은 유닛 수로도 표현력이 커짐 (depth efficiency).
- 우리가 부드럽다고 느끼는 신경망의 곡선은 사실 **ReLU가 만든 직선 조각들의 이음(piecewise linear)**이다.

### 더 해볼 것
- 목표 곡선 `y`를 더 복잡하게 (예: `np.sin(3*x)`) 바꿔보고, 필요한 뉴런 수가 어떻게 달라지는지 관찰
- ReLU를 `torch.tanh`로 바꿔보고 예측 곡선 모양이 어떻게 달라지는지 비교
- `hidden`을 아주 크게(예: 500) 두면 어떻게 되는지 (과적합 조짐도 관찰)
